# KADaP GPUaaS 첫 워크로드 실습 노트북

이 노트북은 KADaP 인공지능 개발 플랫폼(ide.bigdata-car.kr)에서 워크로드를 처음 생성해볼 때 사용하는 실습용 예제입니다.

**진행 순서**
1. GPU/VRAM 환경 확인
2. 텐서 연산 벤치마크 (GPU가 실제로 연산에 쓰이는지 확인)
3. 간단한 신경망 학습 데모 (합성 데이터, 몇 초 내 완료)
4. 학습 결과를 마이디스크에 저장

> 실행 전 확인: 워크로드 생성 시 이미지는 `Built-in Image` 중 PyTorch+CUDA 조합(예: `xiilab/astrago:pytorch-23.07-cuda12.1`)을 선택하고, GPU는 A100 MIG 슬롯(9GB)처럼 가벼운 자원으로도 충분합니다.

## 1. GPU / VRAM 환경 확인

In [ ]:
import torch, subprocess

print("PyTorch 버전:", torch.__version__)
print("CUDA 사용 가능:", torch.cuda.is_available())

if torch.cuda.is_available():
    n = torch.cuda.device_count()
    print(f"할당된 GPU 개수: {n}")
    for i in range(n):
        props = torch.cuda.get_device_properties(i)
        vram_gb = props.total_memory / (1024**3)
        print(f"  GPU {i}: {props.name} | VRAM {vram_gb:.1f} GB")
else:
    print("GPU가 잡히지 않았습니다. 워크로드 생성 시 GPU 노드가 정상 할당됐는지 확인하세요.")

print()
print("--- nvidia-smi 원본 출력 ---")
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)

## 2. 텐서 연산 벤치마크

CPU와 GPU에서 동일한 행렬곱 연산 속도를 비교합니다. GPU가 실제로 훨씬 빠르게 나오면 워크로드가 GPU 자원을 정상적으로 쓰고 있다는 뜻입니다.

In [ ]:
import time

size = 4096
a_cpu = torch.randn(size, size)
b_cpu = torch.randn(size, size)

t0 = time.time()
c_cpu = a_cpu @ b_cpu
cpu_time = time.time() - t0
print(f"CPU 행렬곱({size}x{size}) 소요 시간: {cpu_time*1000:.1f} ms")

if torch.cuda.is_available():
    a_gpu = a_cpu.cuda()
    b_gpu = b_cpu.cuda()
    torch.cuda.synchronize()
    t0 = time.time()
    c_gpu = a_gpu @ b_gpu
    torch.cuda.synchronize()
    gpu_time = time.time() - t0
    print(f"GPU 행렬곱({size}x{size}) 소요 시간: {gpu_time*1000:.1f} ms")
    print(f"GPU가 CPU보다 약 {cpu_time/gpu_time:.1f}배 빠름")

## 3. 간단한 신경망 학습 데모

실제 자동차 데이터 대신, 합성(가상) 데이터로 아주 작은 회귀 모델을 몇 초 안에 학습시켜 봅니다.
KADaP GPU 서버 존이 "DGX(학습 컴퓨터)" 역할을 어떻게 수행하는지 가장 작은 규모로 체험하는 단계입니다.

In [ ]:
import torch.nn as nn
import torch.optim as optim

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("학습에 사용할 디바이스:", device)

# 합성 데이터: y = 3x1 - 2x2 + 노이즈
torch.manual_seed(0)
X = torch.randn(2000, 2, device=device)
y = (3 * X[:, 0] - 2 * X[:, 1] + 0.1 * torch.randn(2000, device=device)).unsqueeze(1)

model = nn.Sequential(nn.Linear(2, 16), nn.ReLU(), nn.Linear(16, 1)).to(device)
optimizer = optim.Adam(model.parameters(), lr=0.01)
loss_fn = nn.MSELoss()

for epoch in range(200):
    optimizer.zero_grad()
    pred = model(X)
    loss = loss_fn(pred, y)
    loss.backward()
    optimizer.step()
    if epoch % 40 == 0:
        print(f"epoch {epoch:3d} | loss {loss.item():.4f}")

print("최종 loss:", loss.item())

## 4. 결과 저장 (마이디스크)

워크로드가 종료되면 설치 패키지와 데이터는 모두 삭제되므로, 보관할 결과물은 종료 전 마이디스크 경로로 옮겨야 합니다.
마이디스크 경로: `root / 자동차데이터플랫폼(KADaP) / MyDisk`

In [ ]:
import os

save_dir = "/root/자동차데이터플랫폼(KADaP)/MyDisk/practice"
os.makedirs(save_dir, exist_ok=True)

save_path = os.path.join(save_dir, "toy_model.pt")
torch.save(model.state_dict(), save_path)
print("모델 저장 완료:", save_path)

## 체크리스트

- [ ] GPU/VRAM이 예상한 사양으로 잡혔는지 확인했다
- [ ] GPU 연산이 CPU보다 빠르게 나오는지 확인했다
- [ ] 간단한 학습이 오류 없이 끝까지 돌았다
- [ ] 결과 파일을 마이디스크로 옮겼다
- [ ] 워크로드를 종료했다 (홈 화면 또는 워크로드 상세에서 종료 버튼)

다음 단계로 넘어갈 때는 이 노트북 대신 실제 KADaP 데이터(자동차 데이터 포털에서 신청한 데이터셋)를 데이터셋 경로로 연결해 학습해보는 것을 추천합니다.